# Party Affiliation Analysis

This notebook analyzes the political party assignments of individuals based on their election history using a Personalized PageRank algorithm.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from main import _setup_context
from analysis.graph import PeopleParties, search_person
from scrapers.pkw.elections import committee_to_party
from analysis.notebooks.runner import run_pipeline

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## Committee to Party Mapping
This shows how various electoral committees are grouped into major political parties.

In [ ]:
mapping_df = pd.DataFrame(list(committee_to_party.items()), columns=["Committee", "Party"])
display(mapping_df.sort_values("Party"))

## Calculate PPR Scores
Running the graph analysis to calculate affiliation scores for all individuals.

In [ ]:
df_scores = run_pipeline(PeopleParties)[1]
print(f"Calculated scores for {len(df_scores)} individuals.")
display(df_scores.head())

In [ ]:
df_scores["original_sum"].sort_values().reset_index().plot()

In [ ]:
df_scores.sum(axis=1).sort_values().reset_index().plot()

## Search for Individuals
Enter a name (or part of it) to see their party affiliation scores.

In [ ]:
def search_and_analyze(query, scores_df):
    matches = search_person(query, scores_df)
    print(f"Found {len(matches)} match(es).")
    
    for name in matches[:3]:  # Limit to 3 visual results
        person_scores = scores_df.loc[name].sort_values(ascending=False)
        person_scores = person_scores[person_scores > 0]
        print(person_scores)
        
        if person_scores.empty:
            print(f"{name.title()} has no calculated affiliations.")
            continue
            
        plt.figure(figsize=(10, 4))
        person_scores.plot(kind='bar', color=plt.cm.Paired(range(len(person_scores))))
        plt.title(f"Affiliation Scores: {name.upper()}", fontsize=12, fontweight='bold')
        plt.ylabel("Score")
        plt.xticks(rotation=45, ha='right')
        plt.ylim(0, 1.15)
        
        # Add text labels on bars
        for i, v in enumerate(person_scores):
            plt.text(i, v + 0.02, f"{v:.2f}", ha='center', fontsize=10, fontweight='bold')
            
        plt.tight_layout()
        plt.show()
        
    if len(matches) > 3:
        print(f"Wait, there are {len(matches)} total matches! Showing details for top 3 above.")
        print(f"Other matching names: {matches[3:15]}")

# Search Examples
search_and_analyze("Donald Tusk", df_scores)
search_and_analyze("Jarosław Kaczyński", df_scores)

## Overall Party Distribution
Number of people assigned to each party based on their highest affiliation score.

In [ ]:
assigned_parties = df_scores[df_scores > 0.01].idxmax(axis=1).dropna()
party_counts = assigned_parties.value_counts()

plt.figure(figsize=(12, 6))
colors = sns.color_palette("husl", len(party_counts))
ax = party_counts.plot(kind='bar', color=colors)
plt.title("People Distribution by Party Affiliation (Highest PPR Score)", fontsize=14)
plt.ylabel("Count", fontsize=12)
plt.xlabel("Party", fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add count labels
for i, v in enumerate(party_counts):
    ax.text(i, v + (max(party_counts)*0.01), f"{v:,}", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

display(party_counts.to_frame("Count"))

## Multi-Party Affiliations
People who have high scores in more than one party (indicating a shift in affiliation or participation in multiple coalitions).

In [ ]:
# People with at least 2 parties having score > 15%
ambiguous = df_scores[(df_scores > 0.15).sum(axis=1) >= 2]
print(f"Found {len(ambiguous)} individuals with multi-party affiliations.")
display(ambiguous.head(20))